In [64]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [ ]:
jhjkh

In [65]:
df=pd.read_csv("flats_house_post_feature_selection.csv")

In [66]:
df.columns

Index(['bedrooms', 'baths', 'area_sqft', 'kitchens', 'store_rooms', 'gym',
       'furnishing_score', 'agePossession', 'property_type',
       'has_servant_room', 'luxury_category', 'floor_category', 'price'],
      dtype='object')

In [78]:
Q1 = df['baths'].quantile(0.25)
Q3 = df['baths'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['baths'] < lower_bound) |
    (df['baths'] > upper_bound)
]

outliers[['baths']].head()

,baths


In [68]:
df = df[
    (df['baths'] >= lower_bound) &
    (df['baths'] <= upper_bound)
]

In [69]:
df.shape

(23062, 13)

In [70]:
df.columns

Index(['bedrooms', 'baths', 'area_sqft', 'kitchens', 'store_rooms', 'gym',
       'furnishing_score', 'agePossession', 'property_type',
       'has_servant_room', 'luxury_category', 'floor_category', 'price'],
      dtype='object')

In [71]:
# df=df.drop(columns=['kitchens','store_rooms','gym','has_servant_room'])

In [72]:
df.columns

Index(['bedrooms', 'baths', 'area_sqft', 'kitchens', 'store_rooms', 'gym',
       'furnishing_score', 'agePossession', 'property_type',
       'has_servant_room', 'luxury_category', 'floor_category', 'price'],
      dtype='object')

In [73]:
df.sample(5)

,bedrooms,baths,area_sqft,kitchens,store_rooms,gym,furnishing_score,agePossession,property_type,has_servant_room,luxury_category,floor_category,price
21269,3.0,4.0,1361.255,1.0,1.0,1,25,1.0,1,1,0.0,1.0,1.70
12957,3.0,4.0,816.753,2.0,1.0,1,13,4.0,1,0,2.0,1.0,1.46
6565,5.0,6.0,5445.020,2.0,1.0,0,25,4.0,1,1,3.0,1.0,8.70
12869,3.0,4.0,816.753,2.0,1.0,0,14,4.0,1,0,2.0,1.0,1.18
22825,6.0,7.0,10890.040,1.0,1.0,1,25,5.0,1,0,3.0,1.0,18.50


In [74]:
df['floor_category'].value_counts()

floor_category
1.0    18572
2.0     4324
0.0      134
3.0       32
Name: count, dtype: int64

In [75]:
df=pd.get_dummies(
    df,
    columns=['agePossession'],
    drop_first=True
)

In [76]:
df.head()

,bedrooms,baths,area_sqft,kitchens,store_rooms,gym,furnishing_score,property_type,has_servant_room,luxury_category,floor_category,price,agePossession_1.0,agePossession_2.0,agePossession_3.0,agePossession_4.0,agePossession_5.0
0,4.0,4.0,3264.0,1.0,1.0,0,37,0,1,3.0,2.0,4.55,True,False,False,False,False
1,4.0,5.0,3318.4,1.0,0.0,0,31,0,1,3.0,2.0,4.50,True,False,False,False,False
2,3.0,4.0,2720.0,1.0,1.0,1,49,0,1,3.0,2.0,3.55,True,False,False,False,False
3,4.0,4.0,2067.2,2.0,1.0,0,18,0,1,3.0,2.0,4.45,True,False,False,False,False
4,4.0,5.0,3318.4,1.0,1.0,0,18,0,1,3.0,2.0,4.65,True,False,False,False,False


In [79]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# numerical columns
num_cols = [
    'bedrooms',
    'baths',
    'area_sqft',
    'kitchens',
    'store_rooms',
    'gym',
    'furnishing_score',
    'property_type',
    'has_servant_room',
    'luxury_category',
    'floor_category',

    'agePossession_1.0',
    'agePossession_2.0',
    'agePossession_3.0',
    'agePossession_4.0',
    'agePossession_5.0'
]

# preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# features and target
X = df[num_cols]
y = df['price']

In [80]:
y = np.log1p(df['price'])

In [81]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

# KFold
kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# cross validation
scores = cross_val_score(
    pipeline,          # your pipeline
    X,                 # features
    y,                 # log transformed target
    cv=kfold,
    scoring='r2'
)

# results
print("R2 Scores:", scores)
print("Mean R2:", scores.mean())

R2 Scores: [0.83821278 0.84866185 0.84055204 0.83036688 0.84233657 0.83837077
 0.83077799 0.85046195 0.84122718 0.84482112]
Mean R2: 0.8405789144085588


In [82]:
scores.std()

np.float64(0.006287552049780777)

In [83]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score

# KFold
kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Ridge pipeline
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

# cross validation
scores = cross_val_score(
    ridge_pipeline,
    X,
    y,
    cv=kfold,
    scoring='r2'
)

# results
print("R2 Scores:", scores)
print("Mean R2:", scores.mean())

R2 Scores: [0.83821395 0.84865916 0.8405542  0.8303657  0.8423375  0.8383723
 0.83077868 0.85046037 0.84122726 0.84482066]
Mean R2: 0.8405789778195544


In [95]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import numpy as np

# features
X = df[num_cols]

# target
y = df['price']

# log transform
y_log = np.log1p(y)

# standard scaling
scaler = StandardScaler()
X = X.drop(columns=['bedrooms'])

X_scaled = scaler.fit_transform(X)

# model
lr = LinearRegression()

# train
lr.fit(X_scaled, y_log)

LinearRegression()

In [96]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_
})

coef_df = coef_df.sort_values(
    by='Coefficient',
    ascending=False
)

coef_df

,Feature,Coefficient
1,area_sqft,0.483391
0,baths,0.189641
5,furnishing_score,0.038727
13,agePossession_4.0,0.029569
10,agePossession_1.0,0.027286
7,has_servant_room,0.013670
14,agePossession_5.0,0.011878
3,store_rooms,0.009614
9,floor_category,0.009215
12,agePossession_3.0,0.005546


In [97]:
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

In [98]:
# add constant
X_with_const = sm.add_constant(X_scaled_df)

# fit model
model = sm.OLS(y_log, X_with_const).fit()

# summary
print(model.summary())

ValueError: The indices for endog and exog are not aligned

In [100]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit OLS model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     8125.
Date:                Wed, 20 May 2026   Prob (F-statistic):               0.00
Time:                        14:59:19   Log-Likelihood:                -2402.9
No. Observations:               23062   AIC:                             4838.
Df Residuals:                   23046   BIC:                             4967.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.6128      0.002    911.752      0.0

In [101]:
# stds
x_std = X.std()
y_std = y_log.std()

# standardized coefficients
beta = lr.coef_

# unstandardized coefficients
b = beta * (y_std / x_std)

# percentage impact
percent_impact = (np.exp(b) - 1) * 100

# dataframe
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Unstandardized_Coefficient': b,
    'Percent_Impact': percent_impact
})

coef_df.sort_values(
    by='Percent_Impact',
    ascending=False
)

,Feature,Unstandardized_Coefficient,Percent_Impact
baths,baths,0.082501,8.599932
agePossession_5.0,agePossession_5.0,0.070940,7.351688
agePossession_4.0,agePossession_4.0,0.044944,4.596964
agePossession_1.0,agePossession_1.0,0.036933,3.762313
agePossession_2.0,agePossession_2.0,0.023069,2.333676
has_servant_room,has_servant_room,0.018881,1.906083
floor_category,floor_category,0.015285,1.540247
store_rooms,store_rooms,0.013522,1.361420
agePossession_3.0,agePossession_3.0,0.012140,1.221422
property_type,property_type,0.003984,0.399226


In [90]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF dataframe
vif_df = pd.DataFrame()

# feature names
vif_df["Feature"] = X.columns

# calculate VIF
vif_df["VIF"] = [
    variance_inflation_factor(X_scaled_df.values, i)
    for i in range(X_scaled_df.shape[1])
]

# sort
vif_df = vif_df.sort_values(
    by="VIF",
    ascending=False
)

vif_df

,Feature,VIF
1,baths,6.120583
0,bedrooms,6.020572
11,agePossession_1.0,5.932664
14,agePossession_4.0,5.444215
7,property_type,3.610855
13,agePossession_3.0,2.973256
6,furnishing_score,2.824252
2,area_sqft,2.019848
5,gym,1.784469
3,kitchens,1.628047


In [ ]:
lr.fit()

In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

# pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ))
])

# KFold
kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# CV
scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=kfold,
    scoring='r2'
)

print("R2 Scores:", scores)
print("Mean R2:", scores.mean())

R2 Scores: [0.88788476 0.88906372 0.89817697 0.88849263 0.89573589 0.89636084
 0.88638661 0.89861234 0.88419824 0.88373233]
Mean R2: 0.8908644329914732


In [28]:
from sklearn.ensemble import RandomForestRegressor

# model
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

# train
rf.fit(X, y)

# feature importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

importance_df

,Feature,Importance
2,area_sqft,0.936562
3,furnishing_score,0.016680
0,bedrooms,0.012134
7,agePossession,0.011439
1,baths,0.010772
5,luxury_category,0.006841
6,floor_category,0.004437
4,property_type,0.001135


In [33]:
ridge = Ridge(alpha=1.0)

ridge.fit(X, y)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': ridge.coef_
})

coef_df.sort_values(by='Coefficient', ascending=False)

,Feature,Coefficient
1,baths,0.118391
6,floor_category,0.020049
0,bedrooms,0.003759
7,agePossession,0.002895
3,furnishing_score,0.002289
2,area_sqft,0.000191
4,property_type,-0.013561
5,luxury_category,-0.021217
